<a href="https://colab.research.google.com/github/nika19du/AI-for-Developers-summer-2026-/blob/main/Copy_of_LangChain_(Part_2).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install -q langchain langchain-openai

In [45]:
from google.colab import userdata
from langchain.messages import AIMessage, HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI
from pydantic import SecretStr

def print_response(response: AIMessage):
    print(f"Response id: {response.id}")
    if response.usage_metadata is not None:
        input_tokens = response.usage_metadata.get("input_tokens", 0);
        cached_tokens = response.usage_metadata.get("input_token_details",{}).get("cache_read", 0)
        output_tokens = response.usage_metadata.get("output_tokens", 0)
        reasoning_tokens = response.usage_metadata.get("output_token_details", {}).get("reasoning", 0)

        print(f"Input tokens: {input_tokens} ({cached_tokens} cached); Output tokens: {output_tokens} ({reasoning_tokens} reasoning)")

    print()
    print(f"{'-' * 20} [Output] {'-' * 20}")
    print(response.text)

In [44]:
openai_api_key = SecretStr(userdata.get('OPENAI_API_KEY'))

# Note: By default ChatOpenAI uses Chat Completions API
#Note that LangChain's OpenAI integration uses the Chat Completions API by default. This behavior can be controlled through the use_responses_api parameter (but keep in mind that the output format may change).
openai_model = ChatOpenAI(model = "gpt-5-nano", api_key = openai_api_key, reasoning_effort = "low", use_responses_api = True)

In [6]:
first_response = museum_audio_guide_response = openai_model.invoke(
    input = [
        HumanMessage("Hello! My name is Nikol and I am from Bulgaria. How are you yoday?")
    ]
)

In [7]:
print_response(first_response)

Response id: lc_run--01a015eb-7cfc-72d0-8fba-8c0f1dcf70d9-0
Input tokens: 24 (0 cached); Output tokens: 116 (64 reasoning)

-------------------- [Output] --------------------
Hi Nikol! I’m here and ready to help. I’m doing well, thanks for asking. How about you?

If you’d prefer, I can reply in Bulgarian. What would you like to do today?


In [8]:
second_response = museum_audio_guide_response = openai_model.invoke(
    input = [
        HumanMessage("What do you remember amout me?")
    ]
)

In [9]:
print_response(second_response)

Response id: lc_run--01a015ec-6c64-7ba0-8a16-86056e926712-0
Input tokens: 14 (0 cached); Output tokens: 176 (64 reasoning)

-------------------- [Output] --------------------
I don’t have memory of past chats unless you share details in this conversation. In this session, I can keep track of what you tell me so far, but I won’t remember it next time.

If you’d like, tell me what you want me to remember or note down (e.g., your interests, goals, preferred name), and I’ll keep track for this chat. You can also ask me to summarize what we’ve discussed so far. What would you like me to remember or help with?


In [10]:
third_reponse = openai_model.invoke(
    input = [
        HumanMessage("Hello! My name is Nikol and I am from Bulgaria. How are you yoday?"),
        first_response,
        HumanMessage("What do you remember amout me?")
    ]
)

In [11]:
print_response(third_reponse)

Response id: lc_run--01a015ee-08b3-7ec2-89b6-8de05e1df0e9-0
Input tokens: 85 (0 cached); Output tokens: 268 (192 reasoning)

-------------------- [Output] --------------------
I remember from this chat that your name is Nikol and you’re from Bulgaria. I don’t retain personal details after our chat ends unless you tell me to. If you’d like, I can keep preferences or notes for the rest of this conversation. Would you like me to respond in Bulgarian or English? How can I help today?


In [31]:
from pydantic import BaseModel

class WorkshopBrief(BaseModel):
    title: str
    audience: str
    duration_minutes: float
    key_takeaways: list[str]
    materials_needed: list[str]


openai_structured_output_model = openai_model.with_structured_output(WorkshopBrief, include_raw=True)

In [15]:
workshop_brief = openai_model.invoke(
    input=[
        SystemMessage("You are an expert event organizer."),
        HumanMessage("Design a beginner-friendly Saturday workshop about balcony herb gardening.")
    ]
)

In [16]:
print_response(workshop_brief)

Response id: resp_0c536ee3ad08be86006a849a63652087d2a2b20c1250829668
Input tokens: 28 (0 cached); Output tokens: 1987 (64 reasoning)

-------------------- [Output] --------------------
Here’s a complete, beginner-friendly Saturday workshop plan for balcony herb gardening. It’s designed for 2–3 hours of content plus setup and wrap-up, suitable for community centers, garden clubs, or urban farms.

1) Overview
- Title: Balcony Herb Gardening: Fresh Herbs, Easy Grow
- Target audience: Complete beginners, urban dwellers with limited space (balconies, patios, or window sills)
- Duration: 3 hours total (including breaks)
- Capacity: 12–20 participants (adjustable)
- Learning goals:
  - Understand the basics of growing common balcony herbs (basil, parsley, chives, thyme, oregano, mint, cilantro)
  - Learn how to choose containers, soil, and light conditions for balconies
  - Practice simple planting, transplanting, and care routines
  - Learn pest prevention, basic troubleshooting, and harvest

In [36]:
workshop_brief2 = openai_structured_output_model.invoke(
    input=[
        SystemMessage("You are an expert event organizer. The `key_takeaways` list must contain exactly three items."),
        HumanMessage("Design a beginner-friendly Saturday workshop about balcony herb gardening.")
    ]
)

In [37]:
print_response(workshop_brief2["raw"])

Response id: resp_0fe082d64f837bbc006a849cfcb49c87d2a222910021468e8d
Input tokens: 139 (0 cached); Output tokens: 269 (64 reasoning)

-------------------- [Output] --------------------
{"title":"Saturday Starter: Balcony Herb Gardening for Beginners","audience":"Urban dwellers and apartment residents new to gardening","duration_minutes":90,"key_takeaways":["How to choose a sunny balcony space and select beginner-friendly herbs","Step-by-step methods for potting, watering, fertilizing, and pest prevention","A simple, scalable care routine and troubleshooting tips for common balcony gardening challenges"],"materials_needed":["Potted herb starter set (basil, parsley, chives) or seeds","3–5 small containers or window boxes","Potting mix and slow-release fertilizer","Watering can or spray bottle","Plant labels and a permanent marker","Trowel or small shovel","Drip tray or saucers","Notebook or printable care sheet"]}


In [38]:
res = workshop_brief2["parsed"]

print(res.model_dump_json(indent=2))

{
  "title": "Saturday Starter: Balcony Herb Gardening for Beginners",
  "audience": "Urban dwellers and apartment residents new to gardening",
  "duration_minutes": 90.0,
  "key_takeaways": [
    "How to choose a sunny balcony space and select beginner-friendly herbs",
    "Step-by-step methods for potting, watering, fertilizing, and pest prevention",
    "A simple, scalable care routine and troubleshooting tips for common balcony gardening challenges"
  ],
  "materials_needed": [
    "Potted herb starter set (basil, parsley, chives) or seeds",
    "3–5 small containers or window boxes",
    "Potting mix and slow-release fertilizer",
    "Watering can or spray bottle",
    "Plant labels and a permanent marker",
    "Trowel or small shovel",
    "Drip tray or saucers",
    "Notebook or printable care sheet"
  ]
}


## Steaming

In [39]:
# стриймваме респонса, дума по дума се подготвя самият отговор
# токън по токън
for chunk in openai_model.stream(
    input = [
        SystemMessage("You are an expert in culinary."),
        HumanMessage("Design a one-evening street-food through Seoul for a curious first-time visitor who wants bold flavors but no seafod.")
    ]
):
  if chunk.text:
      print(chunk.text, end="", flush = True)

Here’s a compact, bold-flavor, seafood-free, one-evening street-food route to get a true Seoul bite-feel. It’s paced for a curious first-timer and keeps you off seafood-heavy items.

Overview
- Neighborhoods: start at Gwangjang Market (central, classic), loop to Jongno area, then swing to Hongdae for a lively end.
- Flow: walk-friendly routes, mostly on flat streets; plan about 3–4 hours of eating plus a drink stop.
- Bold flavors you’ll taste: gochujang (red chili paste), gochugaru (chili flakes), sesame, garlic, rendered pork/duck, crispy textures, smoky sauces.

Stops and what to order

1) Gwangjang Market − “classic Seoul first bite”
- What to try:
  - Bindaetteok (mung bean pancake): crisp edges, garlicky, savory. Gluten-friendly if cooked in clean oil.
  - Mayak gimbap: tiny seaweed rice rolls; addictive and smoky-salty without seafood, if you skip the fish sauce drizzle.
  - Optional: Sundae (Korean blood sausage, typically pork-based): rich, peppery, bold. Great if you’re curio

In [42]:
from langchain_core.messages.content import create_image_block, create_text_block

analyze_image_response = openai_model.invoke(
    input=[
        SystemMessage("You are an expert image analyst. Keep your answer concise and structured."),
        HumanMessage(
            content_blocks=[
                create_text_block("Analyze this image and return a one-sentence summary followed by 5 key visible objects."),
                create_image_block(url="https://freerangestock.com/_next/image?url=%2Fimages%2Fsample%2F88947%2Fpainter-working-in-studio_4460x4460.jpg&w=3840&q=75")
            ]
        )
    ]
)

## Vision

In [43]:
print_response(analyze_image_response)

Response id: resp_090769fe7e44de0f006a84aca563ac87d2a49aac039431ff8b
Input tokens: 879 (0 cached); Output tokens: 259 (192 reasoning)

-------------------- [Output] --------------------
A person is painting at an easel in a sunlit art studio surrounded by various canvases. 
- Wooden easel
- Painter (person)
- Canvas on the easel
- Brush jar with brushes
- Stools/chair and stacked canvases leaning against the wall
